# EcoHome Energy Advisor — Agent Testing & Evaluation

This notebook tests the EcoHome Energy Advisor agent across the five key scenarios
described in the project specification. It also evaluates agent responses on:
- **Accuracy**: Does the response use real data from tools?
- **Specificity**: Does it include numbers (kWh, $, hours)?
- **Actionability**: Does it give clear, actionable recommendations?
- **RAG Usage**: Does it reference energy-saving tips from the knowledge base?

**Prerequisites:** Run `01_db_setup.ipynb` and `02_rag_setup.ipynb` first.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv()

from agent import build_agent, chat
from langchain_core.messages import HumanMessage

print('Building EcoHome Energy Advisor agent...')
advisor = build_agent(model_name='gpt-4o', temperature=0.0)
print('✅ Agent ready!')

## Scenario 1: EV Charging Optimization

> "When should I charge my electric car tomorrow to minimize cost and maximize solar power?"

In [ ]:
q1 = 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?'
answer1 = chat(q1, agent=advisor, verbose=False)
print(answer1)

## Scenario 2: Thermostat Optimization During Price Spike

> "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?"

In [ ]:
q2 = 'What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?'
answer2 = chat(q2, agent=advisor, verbose=False)
print(answer2)

## Scenario 3: Personalized Energy Reduction Tips

> "Suggest three ways I can reduce energy use based on my usage history."

In [ ]:
q3 = 'Suggest three ways I can reduce energy use based on my usage history.'
answer3 = chat(q3, agent=advisor, verbose=False)
print(answer3)

## Scenario 4: Dishwasher Savings Calculation

> "How much can I save by running my dishwasher during off-peak hours?"

In [ ]:
q4 = 'How much can I save by running my dishwasher during off-peak hours?'
answer4 = chat(q4, agent=advisor, verbose=False)
print(answer4)

## Scenario 5: Pool Pump Scheduling with Weather Forecast

> "What's the best time to run my pool pump this week based on the weather forecast?"

In [ ]:
q5 = "What's the best time to run my pool pump this week based on the weather forecast?"
answer5 = chat(q5, agent=advisor, verbose=False)
print(answer5)

## Bonus Scenarios

In [ ]:
bonus_questions = [
    'What is my highest-energy device and how can I reduce its cost?',
    'My solar system generated less than usual last week. What might be the cause and how do I fix it?',
    'Should I run the dryer tonight or wait until the morning? Consider both price and solar.',
]

for q in bonus_questions:
    ans = chat(q, agent=advisor, verbose=False)
    print(f'Q: {q}')
    print(f'A: {ans[:500]}...')
    print('-' * 70)

## Evaluation Summary

In [ ]:
def evaluate_response(question, answer, criteria):
    """Simple keyword-based evaluation heuristic."""
    score = 0
    details = []
    for criterion, keywords in criteria.items():
        found = any(kw.lower() in answer.lower() for kw in keywords)
        score += int(found)
        details.append(f"  {'✅' if found else '❌'} {criterion}")
    return score, len(criteria), details

evaluations = [
    (
        'EV Charging',
        answer1,
        {
            'Mentions specific hours': ['AM', 'PM', ':00', 'hour'],
            'Mentions cost/price': ['$', 'cost', 'rate', 'price', 'kWh'],
            'Mentions solar': ['solar', 'sun', 'generation'],
            'Mentions off-peak': ['off-peak', 'super off-peak', 'overnight', 'night'],
        }
    ),
    (
        'Thermostat',
        answer2,
        {
            'Mentions temperature': ['°F', '°C', 'degree', 'temperature', 'thermostat'],
            'Mentions peak pricing': ['peak', 'price', 'spike', '$'],
            'Mentions pre-cooling': ['pre-cool', 'pre cool', 'before', 'ahead'],
            'Actionable recommendation': ['set', 'raise', 'lower', 'recommend'],
        }
    ),
    (
        'Usage Tips',
        answer3,
        {
            'Three recommendations': ['1.', '2.', '3.', 'First', 'Second', 'Third'],
            'References usage data': ['usage', 'history', 'kWh', 'device'],
            'Includes savings estimate': ['$', 'save', 'saving'],
            'References knowledge base': ['tip', 'practice', 'best'],
        }
    ),
    (
        'Dishwasher Savings',
        answer4,
        {
            'Dollar amount': ['$'],
            'Mentions kWh': ['kWh', 'energy'],
            'Compares peak vs off-peak': ['peak', 'off-peak', 'vs', 'compared'],
            'Annual or monthly savings': ['year', 'month', 'annual', 'monthly'],
        }
    ),
    (
        'Pool Pump Scheduling',
        answer5,
        {
            'Mentions weather': ['weather', 'sunny', 'cloud', 'forecast'],
            'Mentions specific days': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'tomorrow'],
            'Mentions solar hours': ['solar', '10', '11', '12', 'noon', 'midday'],
            'Mentions run duration': ['hour', 'hours', 'run time'],
        }
    ),
]

print('=== Agent Evaluation Results ===')
total_score, total_max = 0, 0
for scenario, answer, criteria in evaluations:
    score, max_score, details = evaluate_response(scenario, answer, criteria)
    total_score += score
    total_max += max_score
    print(f'\n📊 {scenario}: {score}/{max_score}')
    for d in details:
        print(d)

print(f'\n🏆 Overall Score: {total_score}/{total_max} ({100*total_score//total_max}%)')

if total_score / total_max >= 0.8:
    print('\n✅ PASS — Agent meets EcoHome quality standards!')
else:
    print('\n⚠️  NEEDS IMPROVEMENT — Review system prompt and tool responses.')